# Cross-check: the pipeline vs pandas, DuckDB SQL, statsmodels and NumPy

The analyses in `src/` are written in the Python standard library, so the whole
pipeline reruns anywhere with no install and CI can prove it is byte-stable.

This notebook recomputes the headline numbers **with the standard analyst
toolkit** and asserts they match what the pipeline wrote to `analysis/` and
`reports/`. If any `assert` fails, the notebook (and CI) fails.

| Result | Pipeline (`src/`) | Recomputed here with |
| --- | --- | --- |
| Historical CLV by country (real) | `crm_retention.py` | pandas `groupby` **and** DuckDB SQL |
| RFM segments (real) | `crm_retention.py` | pandas `rank` + `np.select` |
| Cohort retention (real) | `crm_retention.py` | pandas `pivot_table` |
| A/B test z, p-value, 95% CI | `analyze_ab_test.py` | `statsmodels` |
| Markov removal-effect attribution | `attribution.py` | NumPy `linalg.solve` |

In [1]:
import json
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from statsmodels.stats.proportion import confint_proportions_2indep, proportions_ztest

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
crm = json.loads((ROOT / "analysis/crm_retention_metrics.json").read_text())
orders = pd.read_csv(ROOT / "data/online_retail_orders.csv", dtype={"customer_id": str},
                     parse_dates=["order_date"])
print(f"{len(orders):,} real orders, {orders.customer_id.nunique():,} customers (GBP)")

36,969 real orders, 5,878 customers (GBP)


## 1. Historical CLV by country — pandas

In [2]:
cust = orders.groupby("customer_id").agg(
    country=("country", "first"), n_orders=("order_id", "count"),
    monetary=("order_value_gbp", "sum"), last=("order_date", "max"))

clv = (cust.groupby("country")
           .agg(customers=("n_orders", "size"), orders=("n_orders", "sum"), revenue=("monetary", "sum"))
           .query("customers >= 30"))
clv["historical_clv_gbp"] = clv.revenue / clv.customers
clv = clv.sort_values("historical_clv_gbp", ascending=False).round(2)

expected = pd.DataFrame(crm["clv_by_country"]["ranked"]).set_index("country")
assert list(clv.index) == list(expected.index)
assert np.allclose(clv.historical_clv_gbp, expected.historical_clv_gbp, atol=0.01)
clv[["customers", "historical_clv_gbp"]]

,customers,historical_clv_gbp
country,,
Germany,107,4039.09
France,93,3801.93
Spain,39,2782.83
United Kingdom,5349,2752.37


## 2. The same number in SQL — DuckDB

In [3]:
sql = f"""
WITH per_customer AS (
    SELECT customer_id, any_value(country) AS country, SUM(order_value_gbp) AS monetary
    FROM read_csv_auto('{(ROOT / "data/online_retail_orders.csv").as_posix()}', types={{'customer_id': 'VARCHAR'}})
    GROUP BY customer_id
)
SELECT country, COUNT(*) AS customers, ROUND(SUM(monetary) / COUNT(*), 2) AS historical_clv_gbp
FROM per_customer
GROUP BY country
HAVING COUNT(*) >= 30
ORDER BY historical_clv_gbp DESC
"""
by_sql = duckdb.sql(sql).df().set_index("country")
assert np.allclose(by_sql.historical_clv_gbp, expected.historical_clv_gbp, atol=0.01)
by_sql

,customers,historical_clv_gbp
country,,
Germany,107,4039.09
France,93,3801.93
Spain,39,2782.83
United Kingdom,5349,2752.37


## 3. RFM segments — pandas

Quintile scores from the average rank (ties share one score), then the same
segment rules as `crm_retention._segment`.

In [4]:
as_of = orders.order_date.max() + pd.Timedelta(days=1)
n = len(cust)

def score(values, higher_is_better=True):
    pos = values.rank(method="average", ascending=not higher_is_better) - 1  # 0-based midrank
    return 5 - np.minimum(4, np.floor(pos * 5 / n)).astype(int)

r = score((as_of - cust["last"]).dt.days, higher_is_better=False)
# np.round rounds half to even (2.5 -> 2, 3.5 -> 4), exactly like Python's round() in the pipeline
fm = np.round((score(cust.n_orders) + score(cust.monetary)) / 2).astype(int)
segment = np.select(
    [(r >= 4) & (fm >= 4), (r >= 3) & (fm >= 3), (r >= 4) & (fm <= 2),
     (r == 3) & (fm <= 3), r == 2, (r <= 1) & (fm >= 4)],
    ["Champions", "Loyal customers", "New / promising", "Needs attention", "At risk", "Can't lose them"],
    default="Hibernating")
counts = pd.Series(segment, index=cust.index).value_counts()

expected_seg = {s["segment"]: s["customers"] for s in crm["rfm"]["segments"]}
assert counts.to_dict() == expected_seg, (counts.to_dict(), expected_seg)
counts.rename("customers").to_frame()

,customers
Champions,1532
At risk,1174
Hibernating,1060
Loyal customers,938
New / promising,584
Needs attention,472
Can't lose them,118


## 4. Cohort retention — pandas `pivot_table`

In [5]:
orders["cohort"] = orders.cohort_month
orders["offset"] = ((orders.order_date.dt.year - orders.cohort.str[:4].astype(int)) * 12
                    + orders.order_date.dt.month - orders.cohort.str[5:].astype(int))
active = (orders[orders.offset.between(0, 6)]
          .pivot_table(index="cohort", columns="offset", values="customer_id", aggfunc="nunique", fill_value=0))
retention = active.div(orders.groupby("cohort").customer_id.nunique(), axis=0).round(4)

expected_ret = pd.DataFrame({c["cohort_month"]: c["retention"] for c in crm["cohort_retention"]["cohorts"]}).T
assert np.allclose(retention.values, expected_ret.values, atol=1e-4)
(retention.head(6) * 100).round(1)  # % of the cohort active in month k

offset,0,1,2,3,4,5,6
cohort,,,,,,,
2009-12,100.0,35.3,33.4,42.5,38.0,35.9,37.7
2010-01,100.0,20.6,31.1,30.6,26.4,30.0,25.8
2010-02,100.0,23.8,22.5,29.1,24.6,20.0,19.2
2010-03,100.0,19.0,23.0,24.2,23.2,20.3,24.6
2010-04,100.0,19.4,19.4,16.3,18.4,22.4,27.6
2010-05,100.0,15.8,16.9,17.3,17.7,25.6,21.3


## 5. A/B test — statsmodels

In [6]:
ab = json.loads((ROOT / "reports/ab_test_marketing_uplift.json").read_text())
a, b = ab["variant_a"], ab["variant_b"]

z, p = proportions_ztest([b["conversions"], a["conversions"]], [b["sessions"], a["sessions"]])
lo, hi = confint_proportions_2indep(b["conversions"], b["sessions"], a["conversions"], a["sessions"],
                                   method="wald", compare="diff")
assert abs(z - ab["z_score"]) < 1e-3 and abs(p - ab["p_value_two_tailed"]) < 1e-5
assert abs(lo - ab["confidence_interval_95"]["lower"]) < 1e-4 and abs(hi - ab["confidence_interval_95"]["upper"]) < 1e-4
print(f"z = {z:.4f}, p = {p:.6f}, 95% CI on uplift = [{lo:.4%}, {hi:.4%}]")

z = 2.7137, p = 0.006654, 95% CI on uplift = [0.2772%, 1.7228%]


## 6. Markov removal effect — NumPy

Absorbing Markov chain: `P(conversion from start) = x[start]` where
`(I - Q) x = r`. Removing a channel sends every transition into it to `(null)`.

In [7]:
paths = pd.read_csv(ROOT / "data/conversion_paths_sample.csv")
paths["steps"] = paths.path.str.split(r"\s*>\s*")
channels = sorted({c for s in paths.steps for c in s})

trans = {}
for s, j, c in zip(paths.steps, paths.journeys, paths.conversions):
    for end, w in (("(conversion)", c), ("(null)", j - c)):
        seq = ["(start)", *s, end]
        for src, dst in zip(seq, seq[1:]):
            trans[(src, dst)] = trans.get((src, dst), 0) + w
T = pd.Series(trans).unstack(fill_value=0)
T = T.div(T.sum(axis=1), axis=0)  # row-normalised transition probabilities

def p_convert(removed=None):
    states = ["(start)"] + [c for c in channels if c != removed]
    Q = T.reindex(index=states, columns=states, fill_value=0).to_numpy()
    r = T.reindex(index=states)["(conversion)"].fillna(0).to_numpy()
    return np.linalg.solve(np.eye(len(states)) - Q, r)[0]

base = p_convert()
removal = pd.Series({c: 1 - p_convert(c) / base for c in channels})
mk = json.loads((ROOT / "analysis/attribution_metrics.json").read_text())["markov"]
assert abs(base - mk["base_conversion_probability"]) < 1e-6
assert np.allclose(removal, [mk["credit"][c]["removal_effect"] for c in channels], atol=1e-6)
(removal / removal.sum() * 100).sort_values(ascending=False).round(1).rename("markov_weight_%").to_frame()

,markov_weight_%
Paid Search,32.0
Email,20.1
Paid Social,16.8
Display,16.3
Organic Search,9.7
Direct,5.1


**All assertions passed:** the dependency-free pipeline and the standard toolkit agree on every headline number.